In [ ]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("/Users/miranda/Documents/GitHub/Sentinel-FYP/data")
out_dir = DATA_DIR / "ghana_parquet"

csv_files = sorted(DATA_DIR.glob("ghana_20*_Q*.csv"))

dfs = []
for f in csv_files:
    name = f.stem  # ghana_2021_Q3
    _, year, quarter = name.split("_")
    df = pd.read_csv(f)
    df["year"] = int(year)
    df["quarter"] = quarter
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)

keep_cols = ["osm_id","fclass","year","quarter","NDVI","NDMI","NDBI","NDWI","BSI"]
df_all = df_all[keep_cols].copy()

keep_classes = ["residential","service","unclassified","trunk","primary","secondary","tertiary"]
df_all = df_all[df_all["fclass"].isin(keep_classes)].copy()

# Write partitioned Parquet by year
df_all.to_parquet(out_dir, index=False, partition_cols=["year"])
print("Saved partitioned parquet to:", out_dir)

In [1]:
# Convert Ghana Q3 cloud40 CSVs to partitioned Parquet
import pandas as pd
from pathlib import Path

DATA_DIR = Path("/Users/miranda/Documents/GitHub/Sentinel-FYP/data/ghana_s2_Q3_cloud40")
OUT_DIR = Path("/Users/miranda/Documents/GitHub/Sentinel-FYP/data/ghana_q3_cloud40_parquet")

csv_files = sorted(DATA_DIR.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No CSV files found in {DATA_DIR}")

dfs = []
for f in csv_files:
    name = f.stem  # e.g. ghana_2021_Q3_cloud40 or similar
    parts = name.split("_")

    # Find year and quarter from filename robustly
    year = next((int(p) for p in parts if p.isdigit() and len(p) == 4), None)
    quarter = next((p for p in parts if p in {"Q1", "Q2", "Q3", "Q4"}), "Q3")

    df = pd.read_csv(f)
    df["year"] = year
    df["quarter"] = quarter
    dfs.append(df)

df_q3 = pd.concat(dfs, ignore_index=True)

# Standardize possible exported column names
rename_map = {
    "NDVI_mean": "NDVI",
    "NDMI_mean": "NDMI",
    "NDBI_mean": "NDBI",
    "NDWI_mean": "NDWI",
    "BSI_mean": "BSI",
}
df_q3 = df_q3.rename(columns=rename_map)

keep_cols = ["osm_id", "fclass", "year", "quarter", "NDVI", "NDMI", "NDBI", "NDWI", "BSI"]
missing = [c for c in keep_cols if c not in df_q3.columns]
if missing:
    raise ValueError(f"Missing expected columns: {missing}")

df_q3 = df_q3[keep_cols].copy()

keep_classes = ["residential", "service", "unclassified", "trunk", "primary", "secondary", "tertiary"]
df_q3 = df_q3[df_q3["fclass"].isin(keep_classes)].copy()

# Convert indices to numeric
for c in ["NDVI", "NDMI", "NDBI", "NDWI", "BSI"]:
    df_q3[c] = pd.to_numeric(df_q3[c], errors="coerce")

# Save partitioned parquet by year
OUT_DIR.mkdir(parents=True, exist_ok=True)
df_q3.to_parquet(OUT_DIR, index=False, partition_cols=["year"])

print("CSV files read:", len(csv_files))
print("Rows saved:", len(df_q3))
print("Saved parquet to:", OUT_DIR)
df_q3.head()


/var/folders/vs/7b7k7q7s1v19prbwdzphbqm00000gn/T/ipykernel_43373/1994286681.py:21: DtypeWarning: Columns (18,22) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f)
/var/folders/vs/7b7k7q7s1v19prbwdzphbqm00000gn/T/ipykernel_43373/1994286681.py:21: DtypeWarning: Columns (18,22) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f)
/var/folders/vs/7b7k7q7s1v19prbwdzphbqm00000gn/T/ipykernel_43373/1994286681.py:21: DtypeWarning: Columns (18,22) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f)
/var/folders/vs/7b7k7q7s1v19prbwdzphbqm00000gn/T/ipykernel_43373/1994286681.py:21: DtypeWarning: Columns (18,22) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f)


CSV files read: 4
Rows saved: 1316416
Saved parquet to: /Users/miranda/Documents/GitHub/Sentinel-FYP/data/ghana_q3_cloud40_parquet


,osm_id,fclass,year,quarter,NDVI,NDMI,NDBI,NDWI,BSI
0,520162721,trunk,2020,Q3,0.106603,0.011584,-0.011584,-0.099767,-0.014010
1,574240962,trunk,2020,Q3,0.168813,-0.046479,0.046479,-0.183105,0.057814
2,574240965,trunk,2020,Q3,0.138374,-0.065356,0.065356,-0.168411,0.078773
3,574240966,trunk,2020,Q3,0.139395,-0.066729,0.066729,-0.168875,0.078862
4,574240968,trunk,2020,Q3,0.168682,-0.048941,0.048941,-0.181104,0.057520


In [2]:
# Load the Q3 cloud40 parquet and merge it with the existing Ghana parquet dataset
import pandas as pd
from pathlib import Path

BASE_PARQUET_DIR = Path("/Users/miranda/Documents/GitHub/Sentinel-FYP/data/ghana_parquet")
Q3_CLOUD40_DIR = Path("/Users/miranda/Documents/GitHub/Sentinel-FYP/data/ghana_q3_cloud40_parquet")

# load both
df_base = pd.read_parquet(BASE_PARQUET_DIR)
df_q3_40 = pd.read_parquet(Q3_CLOUD40_DIR)

# standardize possible column names
rename_map = {
    "NDVI_mean": "NDVI",
    "NDMI_mean": "NDMI",
    "NDBI_mean": "NDBI",
    "NDWI_mean": "NDWI",
    "BSI_mean": "BSI",
}
df_base = df_base.rename(columns=rename_map)
df_q3_40 = df_q3_40.rename(columns=rename_map)

# keep the common analysis columns
keep_cols = ["osm_id", "fclass", "year", "quarter", "NDVI", "NDMI", "NDBI", "NDWI", "BSI"]
df_base = df_base[keep_cols].copy()
df_q3_40 = df_q3_40[keep_cols].copy()

# remove original Q3 rows for 2020-2023 from the base dataset
replace_years = [2020, 2021, 2022, 2023]
df_base_keep = df_base[
    ~((df_base["year"].isin(replace_years)) & (df_base["quarter"] == "Q3"))
].copy()

# combine base + replacement Q3 cloud40 data
df_all_updated = pd.concat([df_base_keep, df_q3_40], ignore_index=True)

# optional: drop exact duplicates if any
df_all_updated = df_all_updated.drop_duplicates(
    subset=["osm_id", "fclass", "year", "quarter"]
).copy()

print("Base rows:", len(df_base))
print("Replacement Q3 rows:", len(df_q3_40))
print("Final merged rows:", len(df_all_updated))

display(
    df_all_updated.groupby(["year", "quarter"]).size().reset_index(name="rows")
    .sort_values(["year", "quarter"])
)


Base rows: 10531328
Replacement Q3 rows: 1316416
Final merged rows: 5265664


/var/folders/vs/7b7k7q7s1v19prbwdzphbqm00000gn/T/ipykernel_43373/971731182.py:47: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_all_updated.groupby(["year", "quarter"]).size().reset_index(name="rows")


,year,quarter,rows
0,2020,Q1,329104
1,2020,Q2,329104
2,2020,Q3,329104
3,2020,Q4,329104
4,2021,Q1,329104
5,2021,Q2,329104
6,2021,Q3,329104
7,2021,Q4,329104
8,2022,Q1,329104
9,2022,Q2,329104
